# GraphSAGE 학습 (HIVE-20)

자기지도 GraphSAGE로 커뮤니티 지식그래프의 **256차원 노드 임베딩**을 학습한다.

**파이프라인**: (로컬) `export_graph.py` → `graph_export.npz` → **(이 노트북, Colab GPU)** → `embeddings.npz` → (로컬) `sage_export.py` → `content.graph_embedding`

- Colab은 로컬 DB(localhost)에 못 닿으므로 **DB 직접 접속 대신 `graph_export.npz`를 업로드**해서 쓴다.
- **순환 차단**: `similar_to` 엣지는 text_embedding 코사인에서 파생되므로 메시지 패싱에는 쓰되 **링크 예측 타깃에서는 제외**한다. 구조 신호인 `belongs_to`/`precedes`만 positive로 학습 → "text 재포장"을 피한다.
- 런타임: 상단 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU**.

In [ ]:
!pip install -q torch-geometric scikit-learn

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score

# graph_export.npz 업로드 (Colab)
try:
    from google.colab import files
    files.upload()  # graph_export.npz 선택
except Exception:
    pass

d = np.load("graph_export.npz")
feat_dim = int(d["meta"][0])
n_content, n_topic = len(d["content_ids"]), len(d["topic_ids"])
print("content", n_content, "topic", n_topic, "feat_dim", feat_dim,
      "| belongs_to", len(d["belongs_to"]), "similar_to", len(d["similar_to"]), "precedes", len(d["precedes"]))

In [ ]:
# 노드: content[0..n_content-1] 다음에 topic[n_content..]. 피처는 같은 공간(둘 다 text_embedding 파생).
x = torch.tensor(np.vstack([d["content_feat"], d["topic_feat"]]), dtype=torch.float)

def to_edge(arr, src_topic=False, dst_topic=False):
    if len(arr) == 0:
        return torch.empty((2, 0), dtype=torch.long)
    s = arr[:, 0].astype(int) + (n_content if src_topic else 0)
    t = arr[:, 1].astype(int) + (n_content if dst_topic else 0)
    return torch.tensor(np.vstack([s, t]), dtype=torch.long)

bt = to_edge(d["belongs_to"], dst_topic=True)   # content -> topic
st = to_edge(d["similar_to"])                    # content -> content
pr = to_edge(d["precedes"], src_topic=True, dst_topic=True)  # topic -> topic

# 메시지 패싱 엣지 = 전부 + 역방향(무방향화)
mp = torch.cat([bt, st, pr], dim=1)
mp = torch.cat([mp, mp.flip(0)], dim=1)
data = Data(x=x, edge_index=mp, num_nodes=x.size(0))
print(data)

In [ ]:
# 링크 예측 타깃: 구조 신호만(belongs_to + precedes). similar_to(=text 파생)는 제외해 순환 차단.
pos = torch.cat([bt, pr], dim=1)
perm = torch.randperm(pos.size(1))
n_val = max(1, int(0.1 * pos.size(1)))
val_pos = pos[:, perm[:n_val]]
train_pos = pos[:, perm[n_val:]]
print("link-pred positives:", pos.size(1), "(train", train_pos.size(1), "/ val", val_pos.size(1), ")")

In [ ]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_dim, hid=256, out=256):
        super().__init__()
        self.proj = torch.nn.Linear(in_dim, hid)
        self.conv1 = SAGEConv(hid, hid)
        self.conv2 = SAGEConv(hid, out)
        self.drop = torch.nn.Dropout(0.3)

    def forward(self, x, edge_index):
        h = F.relu(self.proj(x))
        h = self.drop(F.relu(self.conv1(h, edge_index)))
        h = self.conv2(h, edge_index)
        return F.normalize(h, p=2, dim=1)   # L2 정규화 → 코사인 친화

In [ ]:
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = GraphSAGE(feat_dim).to(dev)
data = data.to(dev)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

def link_loss(z, pos_edges):
    neg = negative_sampling(pos_edges, num_nodes=z.size(0), num_neg_samples=pos_edges.size(1))
    pos_s = (z[pos_edges[0]] * z[pos_edges[1]]).sum(-1)
    neg_s = (z[neg[0]] * z[neg[1]]).sum(-1)
    s = torch.cat([pos_s, neg_s])
    y = torch.cat([torch.ones_like(pos_s), torch.zeros_like(neg_s)])
    return F.binary_cross_entropy_with_logits(s, y), s, y

tp, vp = train_pos.to(dev), val_pos.to(dev)
for ep in range(1, 201):
    model.train(); opt.zero_grad()
    z = model(data.x, data.edge_index)
    loss, _, _ = link_loss(z, tp)
    loss.backward(); opt.step()
    if ep % 20 == 0:
        model.eval()
        with torch.no_grad():
            z = model(data.x, data.edge_index)
            _, s, y = link_loss(z, vp)
            auc = roc_auc_score(y.cpu().numpy(), torch.sigmoid(s).cpu().numpy())
        print(f"epoch {ep:3d} | loss {loss.item():.4f} | val link-AUC {auc:.3f}")

In [ ]:
# content 노드 임베딩만 추출 → embeddings.npz (적재 대상)
model.eval()
with torch.no_grad():
    z = model(data.x, data.edge_index).cpu().numpy()
graph_emb = z[:n_content].astype("float32")
np.savez("embeddings.npz", content_ids=d["content_ids"], graph_emb=graph_emb)
print("saved embeddings.npz", graph_emb.shape)
try:
    from google.colab import files
    files.download("embeddings.npz")
except Exception:
    pass

In [ ]:
# (발표 자료) t-SNE 시각화
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
emb2 = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(z[:n_content])
plt.figure(figsize=(7, 6))
plt.scatter(emb2[:, 0], emb2[:, 1], s=8)
plt.title("GraphSAGE content embeddings (t-SNE)")
plt.show()

## 다음 단계 (로컬)

`embeddings.npz`를 받아 DB에 적재:
```
cd backend && PYTHONPATH=. venv/bin/python -m app.graph.sage_export --in embeddings.npz --dry-run   # 검증
cd backend && PYTHONPATH=. venv/bin/python -m app.graph.sage_export --in embeddings.npz             # 적재
```

**주의**: 현재 그래프는 노드가 적고(~249) `precedes`가 비어 있어 학습 가치가 제한적이다.
Auto-HKG(HIVE-21)로 `precedes`/하위노드가 채워지고 데이터가 늘어난 뒤 학습하는 것을 권장.